In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached qiskit-aer-0.15.1.tar.gz (6.6 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for qiskit-aer (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [156 lines of output]
      /private/var/folders/6x/f2x_0brx5kn1_bhptbrb65xw0000gn/T/pip-build-env-u8zbfc8g/overlay/lib/python3.13/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :

# BB84 Quantum Key Distribution — No Attacker

simulates the BB84 protoco without an attacker


In [2]:
#helper

simulator = BasicSimulator()

def quantum_random_bit():
    """Generate a single random bit by measuring |+>."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # Prepare |+> = H|0>
    qc.measure(0, 0) # Measure in standard basis -> 0 or 1 with prob 1/2
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def quantum_random_bits(n):
    """Generate a list of n random bits using quantum measurement."""
    return [quantum_random_bit() for _ in range(n)]

# Quick sanity check
print("Sample of 10 quantum random bits:", quantum_random_bits(10))

Sample of 10 quantum random bits: [0, 0, 0, 0, 0, 0, 0, 0, 1, 0]


In [3]:
# alice -> encode qubits]
#   alice generates a random sequence of bits
#   for each bit she randomly picks a basis:
#     standard basis (s): 0 -> |0>, 1 -> |1>
#     diagonal basis (d): 0 -> |+>, 1 -> |->
#   she sends the resulting qubits to bob

NUM_QUBITS = 20  # no. of qubits

# alice uses quantum randomness for both her bits and her basis choices
alice_bits   = quantum_random_bits(NUM_QUBITS)  # the secret bit values
alice_bases  = quantum_random_bits(NUM_QUBITS)  # 0 = standard, 1 = diagonal

def alice_encode(bit, basis):
    """
    Alice encodes one bit into a qubit.
    basis=0 (standard): 0->|0>, 1->|1>
    basis=1 (diagonal): 0->|+>, 1>|->
    Returns the QuantumCircuit representing the prepared qubit.
    """
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)          # flip to |1> (for both bases, start with the correct Z state)
    if basis == 1:       # diagonal basis: apply H to get |+> or |->
        qc.h(0)          # H|0> = |+>,  H|1> = |->
    return qc

# alice prepares all her qubits
alice_qubits = [alice_encode(b, bas) for b, bas in zip(alice_bits, alice_bases)]

basis_label = lambda b: 's' if b == 0 else 'd'
print("ALICE")
print("Bits :", alice_bits)
print("Bases:", [basis_label(b) for b in alice_bases])

ALICE
Bits : [0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0]
Bases: ['s', 's', 'd', 's', 's', 'd', 's', 's', 'd', 'd', 'd', 's', 'd', 's', 'd', 'd', 'd', 'd', 'd', 'd']


In [4]:
# bob -> measuring quibits
#   for each qubit bob receives, he randomly picks a basis
#   (standard or diagonal) to measure in
#   if he picks the same basis as Alice -> he gets Alice's bit
#   if he picks the other basis -> he gets a random bit
#
# measuring in diagonal basis:
#   apply H then do a standard measurement
#   H converts |+> -> |0> and |-> -> |1>

bob_bases = quantum_random_bits(NUM_QUBITS)  # 0 = standard, 1 = diagonal

def bob_measure(alice_qc, basis):
    """
    Bob appends his measurement to Alice's prepared qubit circuit.
    basis=0: standard measurement
    basis=1: diagonal measurement = H then standard measure
    Returns the measured bit.
    """
    qc = alice_qc.copy()
    qc.add_register(__import__('qiskit').ClassicalRegister(1))
    if basis == 1:
        qc.h(0)          # diagonal basis: apply H before measuring
    qc.measure(0, 0)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

bob_bits = [bob_measure(qc, bas) for qc, bas in zip(alice_qubits, bob_bases)]

print("BOB")
print("Bases:", [basis_label(b) for b in bob_bases])
print("Bits :", bob_bits)

BOB
Bases: ['d', 's', 's', 'd', 's', 'd', 'd', 's', 'd', 'd', 'd', 'd', 'd', 's', 'd', 's', 'd', 's', 'd', 's']
Bits : [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1]


In [5]:
# SIFTING

matching_indices = [i for i in range(NUM_QUBITS) if alice_bases[i] == bob_bases[i]]

alice_key = [alice_bits[i] for i in matching_indices]
bob_key   = [bob_bits[i]   for i in matching_indices]

print("--- KEY SIFTING (public channel) ---")
print(f"Matching positions ({len(matching_indices)}/{NUM_QUBITS}):", matching_indices)
print()
print("Alice's key:", alice_key)
print("Bob's key:  ", bob_key)

--- KEY SIFTING (public channel) ---
Matching positions (12/20): [1, 4, 5, 7, 8, 9, 10, 12, 13, 14, 16, 18]

Alice's key: [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0]
Bob's key:   [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0]


In [6]:
# ============================================================
# VERIFICATION — Keys must match (no attacker)
# ============================================================
# Without an attacker, Alice's and Bob's keys must be identical
# at all matching positions.

if alice_key == bob_key:
    print("SUCCESS: Alice and Bob share the same key!")
    print("Shared key:", alice_key)
    print(f"Key length: {len(alice_key)} bits (out of {NUM_QUBITS} transmitted)")
else:
    mismatches = [i for i,(a,b) in enumerate(zip(alice_key,bob_key)) if a!=b]
    print("UNEXPECTED ERROR: Keys do not match at positions", mismatches)

print()
print("--- SUMMARY TABLE ---")
print(f"{'Qubit':>5} | {'A bit':>5} | {'A basis':>7} | {'B basis':>7} | {'B bit':>5} | {'Match?':>7} | {'Key bit':>7}")
print("-" * 60)
for i in range(NUM_QUBITS):
    match = alice_bases[i] == bob_bases[i]
    key_bit = str(alice_bits[i]) if match else '-'
    print(f"{i:>5} | {alice_bits[i]:>5} | {basis_label(alice_bases[i]):>7} | {basis_label(bob_bases[i]):>7} | {bob_bits[i]:>5} | {'YES' if match else 'no':>7} | {key_bit:>7}")

SUCCESS: Alice and Bob share the same key!
Shared key: [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0]
Key length: 12 bits (out of 20 transmitted)

--- SUMMARY TABLE ---
Qubit | A bit | A basis | B basis | B bit |  Match? | Key bit
------------------------------------------------------------
    0 |     0 |       s |       d |     1 |      no |       -
    1 |     0 |       s |       s |     0 |     YES |       0
    2 |     1 |       d |       s |     0 |      no |       -
    3 |     1 |       s |       d |     1 |      no |       -
    4 |     0 |       s |       s |     0 |     YES |       0
    5 |     0 |       d |       d |     0 |     YES |       0
    6 |     1 |       s |       d |     0 |      no |       -
    7 |     0 |       s |       s |     0 |     YES |       0
    8 |     1 |       d |       d |     1 |     YES |       1
    9 |     1 |       d |       d |     1 |     YES |       1
   10 |     1 |       d |       d |     1 |     YES |       1
   11 |     1 |       s |       d | 